# Insurance Premium Prediction (SIGNUP-ONLY — new-applicant realistic)
### Hospital Multi-Model AI Diagnostic & Cost Prediction System
### Google Colab version

**Dataset:** downloaded directly from Kaggle via `kagglehub` —
`mohankrishnathalla/medical-insurance-cost-prediction`.

**Target:** `annual_premium`.

**Colab note:** `/content` is wiped when the runtime disconnects. This notebook saves the
trained model, metrics, and figures to Google Drive at the end (Section 9) — don't skip that
step, or you'll lose the results the moment the session ends.


**Feature scope — SIGNUP-ONLY:** restricted to information available about a brand-new
applicant at the moment they apply — demographics, lifestyle, baseline vitals, disclosed
medical history, and the plan they chose. Excludes claims paid, utilization counts, and risk
scores computed during/after the policy period. This is the model that should back Tab 1's
live prediction in the Streamlit app.


**Features requested (38):**
`age, sex, region, urban_rural, income, education, marital_status, employment_status, household_size, dependents, bmi, smoker, alcohol_freq, visits_last_year, hospitalizations_last_3yrs, days_hospitalized_last_3yrs, medication_count, systolic_bp, diastolic_bp, ldl, hba1c, plan_type, network_tier, deductible, copay, policy_term_years, provider_quality, chronic_count, hypertension, diabetes, asthma, copd, cardiovascular_disease, cancer_history, kidney_disease, liver_disease, arthritis, mental_health`

**Important:** this feature list was designed against a specific 100k-row / 54-column dataset
seen earlier in this project. `mohankrishnathalla/medical-insurance-cost-prediction` may or may not have the exact same schema —
Section 1 below checks for that explicitly and tells you exactly what's missing if it doesn't
match, rather than failing confusingly deep in the pipeline.


## 0. Setup

In [ ]:
!pip install -q kagglehub xgboost shap

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42
sns.set_style("whitegrid")

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/figures", exist_ok=True)


## 1. Download the dataset via kagglehub and validate its schema

In [ ]:
import kagglehub

DATASET_SLUG = "mohankrishnathalla/medical-insurance-cost-prediction"
dataset_path = kagglehub.dataset_download(DATASET_SLUG)
print("Downloaded to:", dataset_path)

# Locate the CSV file(s) inside the downloaded folder
csv_files = []
for root, _, files_in_dir in os.walk(dataset_path):
    for f in files_in_dir:
        if f.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("CSV file(s) found:")
for p in csv_files:
    print(" -", p)

assert csv_files, "No CSV found in the downloaded dataset — check DATASET_SLUG."
CSV_PATH = csv_files[0]   # adjust index if multiple CSVs and the wrong one is picked


100%|██████████| 5.62M/5.62M [00:00<00:00, 59.1MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/mohankrishnathalla/medical-insurance-cost-prediction/versions/1
CSV file(s) found:
 - /root/.cache/kagglehub/datasets/mohankrishnathalla/medical-insurance-cost-prediction/versions/1/medical_insurance.csv


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("\nColumns found:")
print(list(df.columns))
df.head()


Shape: (100000, 54)

Columns found:
['person_id', 'age', 'sex', 'region', 'urban_rural', 'income', 'education', 'marital_status', 'employment_status', 'household_size', 'dependents', 'bmi', 'smoker', 'alcohol_freq', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'plan_type', 'network_tier', 'deductible', 'copay', 'policy_term_years', 'policy_changes_last_2yrs', 'provider_quality', 'risk_score', 'annual_medical_cost', 'annual_premium', 'monthly_premium', 'claims_count', 'avg_claim_amount', 'total_claims_paid', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count', 'proc_consult_count', 'proc_lab_count', 'is_high_risk', 'had_major_procedure']


,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0


In [ ]:
REQUIRED_COLUMNS = ['age', 'sex', 'region', 'urban_rural', 'income', 'education', 'marital_status', 'employment_status', 'household_size', 'dependents', 'bmi', 'smoker', 'alcohol_freq', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'plan_type', 'network_tier', 'deductible', 'copay', 'policy_term_years', 'provider_quality', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health'] + ["annual_premium"]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]

if missing:
    print("SCHEMA MISMATCH — this dataset does not have the columns this notebook expects.")
    print(f"Missing ({len(missing)}):", missing)
    print()
    print("This notebook was built against a richer 54-column underwriting-style dataset seen")
    print("earlier in this project. If this Kaggle dataset is the simpler classic 7-column")
    print("version (age/sex/bmi/children/smoker/region/charges), the feature lists above need")
    print("to be rewritten for that schema instead — stop here and adjust Section 1's")
    print("REQUIRED_COLUMNS / the FEATURES lists to match what actually printed above.")
else:
    print("Schema OK — all required columns present.")


Schema OK — all required columns present.


**Stop and check the output above before continuing.** If there's a schema mismatch, the
rest of this notebook will fail on the first reference to a missing column — better to catch
it here with a clear message than debug a cryptic `KeyError` several cells down.


### Handling `alcohol_freq` missing values (if present)

`alcohol_freq` has categories Occasional / Weekly / Daily but no "Never"/"None" in the version
of this dataset seen earlier — missing values there were almost certainly non-drinkers rather
than unknown data, filled as their own explicit category.


In [ ]:
if "alcohol_freq" in df.columns:
    df["alcohol_freq"] = df["alcohol_freq"].fillna("None")
    print(df["alcohol_freq"].value_counts())
else:
    print("No alcohol_freq column in this dataset — skipping.")


alcohol_freq
Occasional    45078
None          30083
Weekly        19833
Daily          5006
Name: count, dtype: int64


## 2. Business Understanding (recap)

This dataset represents a health insurer's underwriting and billing records. The target,
`annual_premium`, is the dollar amount billed to a policyholder per year.

**Business questions this model answers:**
- *"If a new applicant walks in with these attributes, what should we expect to charge them?"*
- *"Which factor most drives that price up or down?"* — supports the explainability
  requirement (FR-3.4) for non-technical hospital/insurance staff.


## 3. Data Understanding (EDA)

In [ ]:
TARGET = "annual_premium"

plt.figure(figsize=(6, 4))
plt.hist(df[TARGET], bins=60, color="#1f6feb")
plt.title(f"Distribution of {TARGET}")
plt.xlabel(TARGET)
plt.tight_layout()
plt.savefig("/content/figures/target_distribution.png", dpi=150)
plt.show()

print(df[TARGET].describe())


count    100000.000000
mean        582.320040
std         399.583722
min         211.670000
25%         352.070000
50%         463.585000
75%         666.697500
max       10962.550000
Name: annual_premium, dtype: float64


In [ ]:
FEATURES = ['age', 'sex', 'region', 'urban_rural', 'income', 'education', 'marital_status', 'employment_status', 'household_size', 'dependents', 'bmi', 'smoker', 'alcohol_freq', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'plan_type', 'network_tier', 'deductible', 'copay', 'policy_term_years', 'provider_quality', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health']
CATEGORICAL = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']
NUMERIC = ['age', 'income', 'household_size', 'dependents', 'bmi', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'deductible', 'copay', 'policy_term_years', 'provider_quality', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health']

corr_cols = NUMERIC + [TARGET]
plt.figure(figsize=(11, 9))
sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation matrix — numeric features vs annual_premium")
plt.tight_layout()
plt.savefig("/content/figures/correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:
print(f"Correlation of numeric features with {TARGET}:")
print(df[NUMERIC + [TARGET]].corr()[TARGET].sort_values(ascending=False))


Correlation of numeric features with annual_premium:
annual_premium                 1.000000
chronic_count                  0.286698
days_hospitalized_last_3yrs    0.222634
hospitalizations_last_3yrs     0.201935
visits_last_year               0.188857
hypertension                   0.147329
systolic_bp                    0.139419
age                            0.127473
mental_health                  0.121460
diastolic_bp                   0.114142
arthritis                      0.111821
medication_count               0.108598
diabetes                       0.104391
hba1c                          0.081799
asthma                         0.079231
cardiovascular_disease         0.076415
copd                           0.066949
cancer_history                 0.053593
liver_disease                  0.050824
bmi                            0.041675
kidney_disease                 0.040838
deductible                     0.022074
ldl                            0.020838
household_size             

## 4. Data Preparation

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", NUMERIC),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), CATEGORICAL),
])

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


Train size: 80000, Test size: 20000


## 5. Modeling — Linear Regression, Random Forest, XGBoost

In [ ]:
candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                             subsample=0.9, colsample_bytree=0.9, n_jobs=-1, random_state=RANDOM_STATE),
}

fitted_pipelines = {}
results = []

for name, estimator in candidates.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    preds = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results.append({"model": name, "rmse": rmse, "mae": mae, "r2": r2})
    print(f"{name:16s}  RMSE={rmse:,.2f}  MAE={mae:,.2f}  R2={r2:.4f}")


LinearRegression  RMSE=357.35  MAE=218.61  R2=0.1976
RandomForest      RMSE=360.86  MAE=220.49  R2=0.1818
XGBoost           RMSE=359.01  MAE=218.88  R2=0.1901


## 6. Evaluation

In [ ]:
results_df = pd.DataFrame(results).sort_values("rmse")
results_df


,model,rmse,mae,r2
0,LinearRegression,357.348878,218.607857,0.197589
2,XGBoost,359.007390,218.875272,0.190124
1,RandomForest,360.855706,220.487102,0.181763


In [ ]:
best_name = results_df.iloc[0]["model"]
best_pipe = fitted_pipelines[best_name]

plt.figure(figsize=(6, 4))
colors = ["#1f6feb" if m == best_name else "#999999" for m in results_df["model"]]
plt.bar(results_df["model"], results_df["rmse"], color=colors)
plt.ylabel("RMSE ($)")
plt.title("Model Comparison — RMSE on Test Set (lower is better)")
plt.tight_layout()
plt.savefig("/content/figures/model_comparison_rmse.png", dpi=150)
plt.show()

print(f"Best model: {best_name}")


Best model: LinearRegression


## 7. Explainability (SHAP)

In [ ]:
preprocessor_fitted = best_pipe.named_steps["preprocess"]
model_fitted = best_pipe.named_steps["model"]

sample_idx = X_test.sample(n=min(2000, len(X_test)), random_state=RANDOM_STATE).index
X_test_sample = X_test.loc[sample_idx]
X_test_transformed = preprocessor_fitted.transform(X_test_sample)
feature_names = preprocessor_fitted.get_feature_names_out()

if best_name == "LinearRegression":
    explainer = shap.LinearExplainer(model_fitted, X_test_transformed)
else:
    explainer = shap.TreeExplainer(model_fitted)

shap_values = explainer.shap_values(X_test_transformed)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

importance_df.head(20)


,feature,mean_abs_shap
17,num__chronic_count,73.090215
7,num__days_hospitalized_last_3yrs,49.112742
64,cat__network_tier_Bronze,45.241982
55,cat__smoker_Never,34.188412
0,num__age,29.623101
53,cat__smoker_Current,26.849717
5,num__visits_last_year,22.240568
67,cat__network_tier_Silver,19.441349
66,cat__network_tier_Platinum,16.920467
4,num__bmi,13.733596


In [ ]:
top_n = 15
top = importance_df.head(top_n)

plt.figure(figsize=(7, 6))
plt.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color="#1f6feb")
plt.xlabel("Mean |SHAP value| (impact on predicted premium, $)")
plt.title(f"Top {top_n} Feature Importances — {best_name}")
plt.tight_layout()
plt.savefig("/content/figures/feature_importance.png", dpi=150)
plt.show()


## 8. Export model + metadata (local /content — will be lost on disconnect)

In [ ]:
joblib.dump({
    "pipeline": best_pipe,
    "feature_order": FEATURES,
    "categorical": CATEGORICAL,
    "numeric": NUMERIC,
    "model_name": best_name,
    "target": TARGET,
}, "/content/models/premium_model.joblib")

with open("/content/reports/model_metrics.json", "w") as f:
    json.dump({"results": results, "best_model": best_name}, f, indent=2, default=float)

importance_df.to_csv("/content/reports/feature_importance.csv", index=False)

print("Saved locally to /content (will be lost if this runtime disconnects).")


Saved locally to /content (will be lost if this runtime disconnects).


## 9. Save to Google Drive — REQUIRED to keep your results

`/content` is wiped when this Colab runtime disconnects. Run this cell to persist the trained
model, metrics, and figures.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = "/content/drive/MyDrive/insurance_premium_output_" + ("full_data" if len(FEATURES) > 40 else "signup_only")
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

import shutil
shutil.copy("/content/models/premium_model.joblib", f"{DRIVE_SAVE_DIR}/premium_model.joblib")
shutil.copy("/content/reports/model_metrics.json", f"{DRIVE_SAVE_DIR}/model_metrics.json")
shutil.copy("/content/reports/feature_importance.csv", f"{DRIVE_SAVE_DIR}/feature_importance.csv")
for fig in ["target_distribution.png", "correlation_heatmap.png", "model_comparison_rmse.png", "feature_importance.png"]:
    shutil.copy(f"/content/figures/{fig}", f"{DRIVE_SAVE_DIR}/{fig}")

print(f"Saved everything to Google Drive: {DRIVE_SAVE_DIR}")


Mounted at /content/drive
Saved everything to Google Drive: /content/drive/MyDrive/insurance_premium_output_signup_only


## 10. Conclusion — SIGNUP-ONLY

This is the realistic "new applicant" model — the one that should back Tab 1's live prediction
in the Streamlit app, since it only uses information genuinely available before a policy
starts.
